In [1]:
!mkdir -p /kaggle/working/edge && cd /kaggle/working/edge && mkdir -p models results onnx_models
%cd /kaggle/working/edge
!pip install -q onnx onnxscript onnxruntime

/kaggle/working/edge


In [2]:
%%writefile common.py
"""
common.py
---------
Small helper code that every other script re-uses:
  * folders and file names
  * CIFAR-10 data loading
  * the ResNet-18 model (made for 32x32 pictures)
  * train / evaluate helpers
  * loading any of the five saved models
"""
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models.quantization import resnet18 as quantizable_resnet18

# ----------------------------------------------------------------------------
# Folders and file names
# ----------------------------------------------------------------------------
ROOT = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(ROOT, "data")
MODELS_DIR = os.path.join(ROOT, "models")
RESULTS_DIR = os.path.join(ROOT, "results")
ONNX_DIR = os.path.join(ROOT, "onnx_models")
for _d in (DATA_DIR, MODELS_DIR, RESULTS_DIR, ONNX_DIR):
    os.makedirs(_d, exist_ok=True)

# The five models of this assignment and the file each one is saved in.
MODEL_FILES = {
    "fp32": "resnet18_fp32.pth",                              # baseline
    "ptq": "resnet18_ptq_int8.pt",                            # post-training quantization
    "qat": "resnet18_qat_int8.pt",                            # quantization-aware training
    "unstructured": "resnet18_unstructured_pruned.pth",       # weights removed one by one
    "structured": "resnet18_structured_pruned.pth",           # whole channels removed
}
Q1_MODELS = ["fp32", "ptq", "qat"]
Q2_MODELS = ["fp32", "unstructured", "structured"]

# EDGE_QUICK=1 makes every script use a tiny part of the data (only to check that the code runs).
QUICK = os.environ.get("EDGE_QUICK") == "1"

# ----------------------------------------------------------------------------
# Speed settings (tuned so the WHOLE pipeline runs in ~10-20 minutes on a laptop CPU,
# e.g. Intel Core i5-1145G7, 4 cores, 16 GB RAM, no GPU).
#   WIDTH        channels of the first ResNet stage (64 = original ResNet-18, 16 = 16x fewer MACs)
#   TRAIN_IMAGES how many of the 50,000 CIFAR-10 training pictures are used for training
#   NUM_THREADS  CPU threads used by PyTorch (= number of physical cores)
# Each one can be changed without editing code, e.g.  set EDGE_WIDTH=32  (Windows)
# ----------------------------------------------------------------------------
WIDTH = int(os.environ.get("EDGE_WIDTH", 16))
TRAIN_IMAGES = int(os.environ.get("EDGE_TRAIN_IMAGES", 20000))
NUM_THREADS = int(os.environ.get("EDGE_THREADS", min(4, os.cpu_count() or 1)))
torch.set_num_threads(NUM_THREADS)

CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)


def model_path(name):
    return os.path.join(MODELS_DIR, MODEL_FILES[name])


def state_dict_path(name):
    """Side file with the plain weights, e.g. resnet18_ptq_int8_state_dict.pth"""
    return os.path.splitext(model_path(name))[0] + "_state_dict.pth"


# ----------------------------------------------------------------------------
# Setup helpers
# ----------------------------------------------------------------------------
def set_seed(seed=42):
    """Fix the random numbers so results can be repeated."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_quant_backend():
    """Pick the quantization 'engine'. fbgemm/x86 = normal PCs, qnnpack = phones/ARM."""
    engines = torch.backends.quantized.supported_engines
    for name in ("fbgemm", "x86", "qnnpack"):
        if name in engines:
            return name
    raise RuntimeError("No quantization backend found on this machine.")


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
# The pictures are kept in memory as small uint8 tensors and every batch is
# cropped / flipped / normalised with fast tensor code. This is much quicker on a
# CPU than the usual PIL + DataLoader workers (which would steal CPU time from training).
_CACHE = {}
_MEAN_T = torch.tensor(MEAN).view(1, 3, 1, 1)
_STD_T = torch.tensor(STD).view(1, 3, 1, 1)


def _cifar_arrays(train):
    """Whole CIFAR-10 split as (uint8 tensor [N,3,32,32], label tensor). Downloads the first time."""
    if train not in _CACHE:
        ds = torchvision.datasets.CIFAR10(DATA_DIR, train=train, download=True)
        x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()   # N,3,32,32 uint8
        y = torch.tensor(ds.targets, dtype=torch.long)
        _CACHE[train] = (x, y)
    return _CACHE[train]


class TensorLoader:
    """Tiny replacement for DataLoader: gives (images, labels) batches, supports len()."""

    def __init__(self, x, y, batch_size, shuffle=False, augment=False):
        self.x, self.y, self.batch_size = x, y, batch_size
        self.shuffle, self.augment = shuffle, augment

    def __len__(self):
        return (len(self.x) + self.batch_size - 1) // self.batch_size

    def _augment(self, xb):
        """Random crop (padding 4) + random horizontal flip, same as the classic CIFAR recipe."""
        n = xb.size(0)
        padded = F.pad(xb, (4, 4, 4, 4))                                  # zero (black) border
        ox = torch.randint(0, 9, (n,)).tolist()
        oy = torch.randint(0, 9, (n,)).tolist()
        xb = torch.stack([padded[i, :, oy[i]:oy[i] + 32, ox[i]:ox[i] + 32] for i in range(n)])
        flip = torch.rand(n) < 0.5
        return torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)

    def __iter__(self):
        n = len(self.x)
        order = torch.randperm(n) if self.shuffle else torch.arange(n)
        for i in range(0, n, self.batch_size):
            idx = order[i:i + self.batch_size]
            xb = self.x[idx]
            if self.augment:
                xb = self._augment(xb)
            xb = (xb.float().div_(255) - _MEAN_T) / _STD_T
            yield xb, self.y[idx]


def get_train_loader(batch_size=128, num_workers=0):
    """Training pictures (random subset of TRAIN_IMAGES, fixed for every script) with augmentation."""
    x, y = _cifar_arrays(True)
    n = 2000 if QUICK else TRAIN_IMAGES
    if n < len(x):
        idx = torch.randperm(len(x), generator=torch.Generator().manual_seed(123))[:n]
        x, y = x[idx], y[idx]
    return TensorLoader(x, y, batch_size, shuffle=True, augment=True)


def get_test_loader(batch_size=200, num_workers=0):
    x, y = _cifar_arrays(False)
    if QUICK:
        x, y = x[:500], y[:500]
    return TensorLoader(x, y, batch_size)


def get_calibration_loader(num_images=1000, batch_size=100):
    """A small random piece of the TRAINING set (no random crops) used for calibration."""
    x, y = _cifar_arrays(True)
    idx = torch.randperm(len(x), generator=torch.Generator().manual_seed(0))[:num_images]
    return TensorLoader(x[idx], y[idx], batch_size)


# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
def build_model(num_classes=10, width=None):
    """ResNet-18 changed for small 32x32 pictures (3x3 first layer, no max-pool).
    `width` = channels of the first stage (64 = original ResNet-18). The four stages use
    width, 2*width, 4*width, 8*width channels. Default comes from WIDTH at the top of this file.
    We use torchvision's 'quantizable' version so the same class works for
    normal training, PTQ and QAT."""
    w = WIDTH if width is None else width
    model = quantizable_resnet18(weights=None, quantize=False, num_classes=num_classes)
    block = type(model.layer1[0])
    model.inplanes = w
    model.conv1 = nn.Conv2d(3, w, kernel_size=3, stride=1, padding=1, bias=False)
    model.bn1 = nn.BatchNorm2d(w)
    model.maxpool = nn.Identity()
    model.layer1 = model._make_layer(block, w, 2)
    model.layer2 = model._make_layer(block, 2 * w, 2, stride=2)
    model.layer3 = model._make_layer(block, 4 * w, 2, stride=2)
    model.layer4 = model._make_layer(block, 8 * w, 2, stride=2)
    model.fc = nn.Linear(8 * w, num_classes)
    for m in model.modules():                      # same initialisation as torchvision
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
    return model


def get_blocks(model):
    """All 8 residual blocks of ResNet-18."""
    return [b for layer in (model.layer1, model.layer2, model.layer3, model.layer4) for b in layer]


def shrink_block(block, k):
    """Give a residual block a thinner middle: conv1 -> k channels, bn1 -> k, conv2 takes k.
    The block's input and output sizes stay the same, so the skip connection still fits."""
    in_ch, stride = block.conv1.in_channels, block.conv1.stride
    out_ch = block.conv2.out_channels
    block.conv1 = nn.Conv2d(in_ch, k, 3, stride=stride, padding=1, bias=False)
    block.bn1 = nn.BatchNorm2d(k)
    block.conv2 = nn.Conv2d(k, out_ch, 3, stride=1, padding=1, bias=False)


def build_structured_model(keep_counts):
    """Re-create the thin (structured-pruned) architecture from the list of channel counts."""
    model = build_model()
    for block, k in zip(get_blocks(model), keep_counts):
        shrink_block(block, k)
    return model


def load_model(name):
    """Load any of the five models, ready for CPU inference."""
    path = model_path(name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run the script that creates the '{name}' model first.")
    if name in ("fp32", "unstructured"):
        model = build_model()
        model.load_state_dict(torch.load(path, map_location="cpu"))
    elif name == "structured":
        ckpt = torch.load(path, map_location="cpu")
        model = build_structured_model(ckpt["keep_counts"])
        model.load_state_dict(ckpt["state_dict"])
    else:  # ptq / qat are saved as TorchScript files (they run without our class code)
        model = torch.jit.load(path, map_location="cpu")
    return model.eval()


# ----------------------------------------------------------------------------
# Train / evaluate
# ----------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device="cpu"):
    """Returns accuracy in percent."""
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += x.size(0)
    return 100.0 * correct / total


def count_params(model):
    """(total number of weights, number of weights that are not zero)"""
    total = nonzero = 0
    for p in model.parameters():
        total += p.numel()
        nonzero += int((p != 0).sum().item())
    return total, nonzero


def count_macs(model):
    """Count multiply-adds for one 32x32 picture (convolution + linear layers only)."""
    macs = [0]

    def hook(module, inp, out):
        if isinstance(module, nn.Conv2d):
            k = module.kernel_size[0] * module.kernel_size[1]
            macs[0] += out.numel() * (module.in_channels // module.groups) * k
        elif isinstance(module, nn.Linear):
            macs[0] += module.in_features * module.out_features

    handles = [m.register_forward_hook(hook) for m in model.modules()
               if isinstance(m, (nn.Conv2d, nn.Linear))]
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, 3, 32, 32))
    for h in handles:
        h.remove()
    return macs[0]


def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)


Writing common.py


In [3]:
%%writefile q1_train_baseline.py
"""
q1_train_baseline.py
--------------------
STEP 1 of the assignment: train the normal (FP32) ResNet-18 on CIFAR-10 and save it.
This same model is the starting point for PTQ, QAT and both pruning methods.

Run:  python q1_train_baseline.py --epochs 12
"""
import argparse
import csv
import os
import time

import torch

from common import (RESULTS_DIR, build_model, evaluate, get_device, get_test_loader,
                    get_train_loader, model_path, set_seed, train_one_epoch)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=12, help="how many times to go over the training pictures")
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=0.1, help="highest learning rate")
    args = parser.parse_args()

    set_seed(42)
    device = get_device()
    print(f"Training on: {device}")

    train_loader = get_train_loader(args.batch_size)
    test_loader = get_test_loader()

    model = build_model().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9,
                                weight_decay=5e-4, nesterov=True)
    # learning rate goes up then slowly down - this usually gives good accuracy quickly
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=args.lr, epochs=args.epochs, steps_per_epoch=len(train_loader))

    log_path = os.path.join(RESULTS_DIR, "train_log_fp32.csv")
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_acc", "test_acc", "seconds"])
        for epoch in range(1, args.epochs + 1):
            start = time.time()
            loss, train_acc = train_one_epoch(model, train_loader, optimizer, device, scheduler)
            test_acc = evaluate(model, test_loader, device)
            secs = time.time() - start
            print(f"Epoch {epoch:02d}/{args.epochs}  loss {loss:.3f}  "
                  f"train acc {train_acc:.2f}%  test acc {test_acc:.2f}%  ({secs:.0f}s)")
            writer.writerow([epoch, f"{loss:.4f}", f"{train_acc:.2f}", f"{test_acc:.2f}", f"{secs:.1f}"])

    torch.save(model.cpu().state_dict(), model_path("fp32"))
    print(f"\nSaved FP32 model -> {model_path('fp32')}")
    print(f"Training log      -> {log_path}")


if __name__ == "__main__":
    main()


Writing q1_train_baseline.py


In [4]:
%%writefile q1_ptq.py
"""
q1_ptq.py
---------
Post-Training Static Quantization (PTQ).
We take the finished FP32 model and shrink it to INT8 WITHOUT training again.

Steps inside this file:
  1. load the FP32 model
  2. fuse layers (Conv + BatchNorm + ReLU become one layer)
  3. choose the quantization backend
  4. add 'observers' that watch the numbers
  5. calibrate: run ~1000 training pictures through the model so the observers learn the number ranges
  6. convert to INT8
  7. test it and save it

Run:  python q1_ptq.py
"""
import argparse

import torch
from torch.ao.quantization import convert, get_default_qconfig, prepare

from common import (build_model, evaluate, file_size_mb, get_calibration_loader, get_quant_backend,
                    get_test_loader, model_path, set_seed, state_dict_path)


def calibrate(model, loader):
    """Just push pictures through the model. No learning happens here."""
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            model(x)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--calib-images", type=int, default=1000, help="how many training images for calibration")
    args = parser.parse_args()

    set_seed(42)
    test_loader = get_test_loader()

    # 1. load the FP32 model (on CPU - quantized models only run on CPU)
    model = build_model()
    model.load_state_dict(torch.load(model_path("fp32"), map_location="cpu"))
    model.eval()
    fp32_acc = evaluate(model, test_loader)
    print(f"FP32 accuracy: {fp32_acc:.2f}%")

    # 2. fuse compatible layers (Conv+BN+ReLU, Conv+BN)
    model.fuse_model()
    print("Layers fused.")

    # 3. quantization backend
    backend = get_quant_backend()
    torch.backends.quantized.engine = backend
    model.qconfig = get_default_qconfig(backend)
    print(f"Quantization backend: {backend}")

    # 4. insert observers
    prepare(model, inplace=True)

    # 5. calibration with a subset of the TRAINING data
    calib_loader = get_calibration_loader(args.calib_images)
    calibrate(model, calib_loader)
    print(f"Calibrated with {args.calib_images} training images.")

    # 6. convert to INT8
    convert(model, inplace=True)
    print("Converted to INT8.")

    # 7. test + save
    int8_acc = evaluate(model, test_loader)
    print(f"PTQ INT8 accuracy: {int8_acc:.2f}%  (FP32 was {fp32_acc:.2f}%)")

    # TorchScript file = a ready-to-run file that needs no model class code
    example = torch.randn(1, 3, 32, 32)
    scripted = torch.jit.trace(model, example)
    torch.jit.save(scripted, model_path("ptq"))
    torch.save(model.state_dict(), state_dict_path("ptq"))
    print(f"Saved INT8 model -> {model_path('ptq')}  ({file_size_mb(model_path('ptq')):.2f} MB)")


if __name__ == "__main__":
    main()


Writing q1_ptq.py


In [5]:
%%writefile q1_qat.py
"""
q1_qat.py
---------
Quantization-Aware Training (QAT).
Here the model PRACTISES being INT8 while it trains, so it loses less accuracy
than PTQ. 'Fake quantization' modules round the numbers during training.

Steps inside this file:
  1. load the pretrained FP32 model
  2. fuse layers
  3. insert fake-quantization modules (prepare_qat)
  4. fine-tune for a few epochs
  5. convert to a real INT8 model
  6. test it and save it

Run:  python q1_qat.py --epochs 3
"""
import argparse
import os

import torch
from torch.ao.nn.intrinsic.qat import freeze_bn_stats
from torch.ao.quantization import convert, disable_observer, get_default_qat_qconfig, prepare_qat

from common import (MODELS_DIR, build_model, evaluate, file_size_mb, get_device, get_quant_backend,
                    get_test_loader, get_train_loader, model_path, set_seed, state_dict_path, train_one_epoch)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--lr", type=float, default=0.01)
    parser.add_argument("--batch-size", type=int, default=128)
    args = parser.parse_args()

    set_seed(42)
    device = get_device()
    train_loader = get_train_loader(args.batch_size)
    test_loader = get_test_loader()

    # 1. pretrained FP32 model
    model = build_model()
    model.load_state_dict(torch.load(model_path("fp32"), map_location="cpu"))

    # 2 + 3. fuse, set the QAT settings, insert fake-quant modules
    model.train()
    model.fuse_model(is_qat=True)
    backend = get_quant_backend()
    torch.backends.quantized.engine = backend
    model.qconfig = get_default_qat_qconfig(backend)
    prepare_qat(model, inplace=True)
    print(f"QAT prepared (backend: {backend}). Fine-tuning on: {device}")
    model.to(device)

    # 4. fine-tune
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    for epoch in range(1, args.epochs + 1):
        if epoch > max(1, args.epochs // 2):
            model.apply(freeze_bn_stats)   # stop BatchNorm from changing
        if epoch > max(2, (3 * args.epochs) // 4):
            model.apply(disable_observer)  # stop the number ranges from moving
        loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
        scheduler.step()
        print(f"QAT epoch {epoch}/{args.epochs}  loss {loss:.3f}  train acc {train_acc:.2f}%")

    # 5. convert to a real INT8 model (must be done on CPU)
    model.cpu().eval()
    int8_model = convert(model, inplace=False)
    int8_acc = evaluate(int8_model, test_loader)
    print(f"QAT INT8 accuracy: {int8_acc:.2f}%")

    # 6. save: TorchScript (runs anywhere) + state dict
    scripted = torch.jit.trace(int8_model, torch.randn(1, 3, 32, 32))
    torch.jit.save(scripted, model_path("qat"))
    torch.save(int8_model.state_dict(), state_dict_path("qat"))
    print(f"Saved INT8 model -> {model_path('qat')}  ({file_size_mb(model_path('qat')):.2f} MB)")

    # also keep the fake-quant model: export_onnx.py turns it into an INT8 ONNX file
    torch.save(model.state_dict(), os.path.join(MODELS_DIR, "resnet18_qat_fakequant_state_dict.pth"))


if __name__ == "__main__":
    main()


Writing q1_qat.py


In [6]:
%%writefile q2_prune_unstructured.py
"""
q2_prune_unstructured.py
------------------------
Unstructured (magnitude-based) pruning.
Idea: the weights with the smallest size (closest to zero) matter least, so we set them to zero.

Steps inside this file:
  1. load the FP32 model
  2. repeat several ROUNDS:
        a. prune: zero out the smallest weights (in all Conv and Linear layers together)
        b. fine-tune a little so the model recovers
  3. make the pruning permanent (remove the masks)
  4. test and save

Run:  python q2_prune_unstructured.py --rounds 3 --amount 0.45 --ft-epochs 1
"""
import argparse

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

from common import (build_model, count_params, evaluate, get_device, get_test_loader,
                    get_train_loader, model_path, set_seed, train_one_epoch)


def sparsity_percent(model):
    total, nonzero = count_params(model)
    return 100.0 * (1 - nonzero / total)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rounds", type=int, default=3, help="how many pruning rounds")
    parser.add_argument("--amount", type=float, default=0.45,
                        help="fraction of the REMAINING weights removed in each round")
    parser.add_argument("--ft-epochs", type=int, default=1, help="fine-tune epochs after each round")
    parser.add_argument("--final-epochs", type=int, default=2, help="extra fine-tune epochs at the end")
    parser.add_argument("--lr", type=float, default=0.01)
    args = parser.parse_args()

    set_seed(42)
    device = get_device()
    train_loader = get_train_loader()
    test_loader = get_test_loader()

    model = build_model()
    model.load_state_dict(torch.load(model_path("fp32"), map_location="cpu"))
    model.to(device)
    print(f"Original accuracy: {evaluate(model, test_loader, device):.2f}%")

    # which weights can be pruned: every Conv2d and Linear layer
    params_to_prune = [(m, "weight") for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]

    for r in range(1, args.rounds + 1):
        # a. prune: remove the smallest weights (compared across the whole network)
        prune.global_unstructured(params_to_prune, pruning_method=prune.L1Unstructured, amount=args.amount)
        # (masks are applied: weight = weight_orig * mask)
        acc_after_prune = evaluate(model, test_loader, device)
        print(f"Round {r}/{args.rounds}: sparsity {sparsity_percent(model):.1f}%  "
              f"accuracy right after pruning {acc_after_prune:.2f}%")

        # b. fine-tune (masked weights stay zero)
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=1e-4)
        for _ in range(args.ft_epochs):
            train_one_epoch(model, train_loader, optimizer, device)
        print(f"          accuracy after fine-tuning {evaluate(model, test_loader, device):.2f}%")

    # extra fine-tuning at the end with a smaller learning rate
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr / 5, momentum=0.9, weight_decay=1e-4)
    for _ in range(args.final_epochs):
        train_one_epoch(model, train_loader, optimizer, device)

    # 3. make it permanent: delete the mask and keep the zeros inside the weights
    for module, name in params_to_prune:
        prune.remove(module, name)

    final_acc = evaluate(model, test_loader, device)
    total, nonzero = count_params(model)
    print(f"\nFINAL: accuracy {final_acc:.2f}%  sparsity {sparsity_percent(model):.1f}%  "
          f"({nonzero:,} of {total:,} weights are non-zero)")

    torch.save(model.cpu().state_dict(), model_path("unstructured"))
    print(f"Saved -> {model_path('unstructured')}")


if __name__ == "__main__":
    main()


Writing q2_prune_unstructured.py


In [7]:
%%writefile q2_prune_structured.py
"""
q2_prune_structured.py
----------------------
Structured CHANNEL pruning.
Instead of zeroing single weights, we delete whole channels (whole filters).
Then we REBUILD a smaller network, so it really has fewer numbers to compute.

How we keep the shapes correct in ResNet-18:
  Every residual block looks like:   input -> conv1 -> bn1 -> relu -> conv2 -> bn2 -> (+ input)
  The 'middle' channels (conv1 output / bn1 / conv2 input) are private to the block,
  so we can shrink them safely. The block's input and output size never change,
  so the skip connection (+ input) still fits.

  Which channels do we delete? The ones whose filters have the smallest total
  size (L1-norm) - they matter least.

Steps inside this file:
  1. load the FP32 model
  2. for each of the 8 blocks: score conv1's filters, keep the best ones,
     build smaller conv1 / bn1 / conv2 and copy the kept weights in
  3. fine-tune the smaller model
  4. test and save

Run:  python q2_prune_structured.py --ratio 0.5 --ft-epochs 4
"""
import argparse

import torch

from common import (build_model, count_macs, count_params, evaluate, get_blocks, get_device,
                    get_test_loader, get_train_loader, model_path, set_seed, shrink_block,
                    train_one_epoch)


def prune_block_channels(block, ratio):
    """Remove `ratio` of the middle channels of one residual block. Returns how many were kept."""
    old_conv1, old_bn1, old_conv2 = block.conv1, block.bn1, block.conv2
    n_out = old_conv1.out_channels
    keep = max(1, int(round(n_out * (1 - ratio))))

    # importance of each channel = sum of absolute weights of its filter (L1-norm)
    importance = old_conv1.weight.data.abs().sum(dim=(1, 2, 3))
    keep_idx = torch.argsort(importance, descending=True)[:keep].sort().values

    shrink_block(block, keep)   # new, thinner layers (random numbers for now)
    # copy the survivors in
    block.conv1.weight.data = old_conv1.weight.data[keep_idx].clone()
    block.bn1.weight.data = old_bn1.weight.data[keep_idx].clone()
    block.bn1.bias.data = old_bn1.bias.data[keep_idx].clone()
    block.bn1.running_mean = old_bn1.running_mean[keep_idx].clone()
    block.bn1.running_var = old_bn1.running_var[keep_idx].clone()
    block.conv2.weight.data = old_conv2.weight.data[:, keep_idx].clone()
    return keep


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--ratio", type=float, default=0.5, help="fraction of middle channels to delete in each block")
    parser.add_argument("--ft-epochs", type=int, default=4, help="fine-tune epochs after pruning")
    parser.add_argument("--lr", type=float, default=0.02)
    args = parser.parse_args()

    set_seed(42)
    device = get_device()
    train_loader = get_train_loader()
    test_loader = get_test_loader()

    model = build_model()
    model.load_state_dict(torch.load(model_path("fp32"), map_location="cpu"))
    orig_total, _ = count_params(model)
    orig_macs = count_macs(model)
    print(f"Original: {orig_total:,} weights, {orig_macs / 1e6:.1f} M multiply-adds, "
          f"accuracy {evaluate(model, test_loader):.2f}%")

    # 2. prune every block
    keep_counts = [prune_block_channels(b, args.ratio) for b in get_blocks(model)]
    print(f"Channels kept in the middle of each block: {keep_counts}")
    new_total, _ = count_params(model)
    new_macs = count_macs(model)
    print(f"Pruned:   {new_total:,} weights, {new_macs / 1e6:.1f} M multiply-adds "
          f"({100 * (1 - new_macs / orig_macs):.0f}% fewer)")
    print(f"Accuracy right after pruning (before fine-tuning): {evaluate(model, test_loader):.2f}%")

    # 3. fine-tune
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.ft_epochs)
    for epoch in range(1, args.ft_epochs + 1):
        loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
        scheduler.step()
        print(f"Fine-tune epoch {epoch}/{args.ft_epochs}  loss {loss:.3f}  "
              f"test acc {evaluate(model, test_loader, device):.2f}%")

    # 4. save: the list of channel counts + the weights (so the thin model can be rebuilt)
    torch.save({"keep_counts": keep_counts, "state_dict": model.cpu().state_dict()}, model_path("structured"))
    print(f"\nFINAL accuracy {evaluate(model.cpu(), test_loader):.2f}%   Saved -> {model_path('structured')}")


if __name__ == "__main__":
    main()


Writing q2_prune_structured.py


In [8]:
%%writefile export_onnx.py
"""
export_onnx.py
--------------
Turn the models into ONNX files (a format that runs on many devices with ONNX Runtime).

Files created in onnx_models/:
  resnet18_fp32.onnx               from the baseline
  resnet18_unstructured.onnx       from the unstructured-pruned model
  resnet18_structured.onnx         from the structured-pruned model (smaller network)
  resnet18_int8_ort.onnx           INT8 version made by ONNX Runtime static quantization
                                   (it uses calibration pictures, like PTQ)
  resnet18_qat_int8.onnx           the QAT model (with its fake-quant nodes) - only if export works

If one export fails, the script prints why and continues with the others.

Run:  python export_onnx.py
"""
import os

import numpy as np
import torch
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat

from common import (MODELS_DIR, ONNX_DIR, build_model, get_calibration_loader, get_quant_backend,
                    load_model, set_seed)


def export_float_model(model, out_path):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32)
    kwargs = dict(input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
    try:
        torch.onnx.export(model, dummy, out_path, dynamo=False, **kwargs)   # classic exporter
    except TypeError:   # older PyTorch has no 'dynamo' option
        torch.onnx.export(model, dummy, out_path, **kwargs)
    print(f"  saved {out_path}")


class CalibrationReader:
    """Feeds calibration pictures to ONNX Runtime, one batch at a time."""
    def __init__(self, loader):
        self.batches = iter([x.numpy() for x, _ in loader])

    def get_next(self):
        batch = next(self.batches, None)
        return None if batch is None else {"image": batch}


def make_int8_with_onnxruntime(fp32_onnx, out_path):
    from onnxruntime.quantization import QuantFormat, QuantType, quantize_static
    quantize_static(fp32_onnx, out_path, CalibrationReader(get_calibration_loader(1000)),
                    quant_format=QuantFormat.QDQ, activation_type=QuantType.QUInt8,
                    weight_type=QuantType.QInt8, per_channel=True)
    print(f"  saved {out_path}")


def export_qat_fake_quant(out_path):
    """Rebuild the QAT model (with fake-quant modules), load the trained weights, export."""
    sd_path = os.path.join(MODELS_DIR, "resnet18_qat_fakequant_state_dict.pth")
    model = build_model()
    model.train()
    model.fuse_model(is_qat=True)
    model.qconfig = get_default_qat_qconfig(get_quant_backend())
    prepare_qat(model, inplace=True)
    model.load_state_dict(torch.load(sd_path, map_location="cpu"))
    export_float_model(model.eval(), out_path)


def attempt(title, function, *args):
    print(f"\n{title}")
    try:
        function(*args)
    except Exception as err:   # keep going - report the problem clearly
        print(f"  FAILED: {type(err).__name__}: {str(err)[:300]}")


def main():
    set_seed(42)
    fp32_onnx = os.path.join(ONNX_DIR, "resnet18_fp32.onnx")

    attempt("FP32 baseline -> ONNX", lambda: export_float_model(load_model("fp32"), fp32_onnx))
    attempt("Unstructured-pruned -> ONNX",
            lambda: export_float_model(load_model("unstructured"), os.path.join(ONNX_DIR, "resnet18_unstructured.onnx")))
    attempt("Structured-pruned -> ONNX",
            lambda: export_float_model(load_model("structured"), os.path.join(ONNX_DIR, "resnet18_structured.onnx")))
    attempt("INT8 with ONNX Runtime static quantization", make_int8_with_onnxruntime,
            fp32_onnx, os.path.join(ONNX_DIR, "resnet18_int8_ort.onnx"))
    attempt("QAT (fake-quant) model -> ONNX", export_qat_fake_quant,
            os.path.join(ONNX_DIR, "resnet18_qat_int8.onnx"))

    print("\nONNX files now in onnx_models/:", sorted(os.listdir(ONNX_DIR)))


if __name__ == "__main__":
    main()


Writing export_onnx.py


In [9]:
%%writefile benchmark.py
"""
benchmark.py
------------
Automatically tests the models on the full CIFAR-10 test set (10,000 pictures)
and measures: accuracy, file size, number of weights, speed.

  python benchmark.py --group q1     -> fp32, ptq, qat
  python benchmark.py --group q2     -> fp32, unstructured, structured
  python benchmark.py                -> both

Output files (in the results/ folder):
  q1_predictions.csv, q1_benchmark.csv
  q2_predictions.csv, q2_benchmark.csv
"""
import argparse
import csv
import os
import time
import zipfile

import numpy as np
import torch

from common import (CLASSES, MODEL_FILES, Q1_MODELS, Q2_MODELS, RESULTS_DIR, count_macs, count_params,
                    file_size_mb, get_quant_backend, get_test_loader, load_model, model_path, set_seed)


@torch.no_grad()
def all_predictions(model, loader):
    """Run the whole test set. Returns labels, predictions, confidences (numpy arrays)."""
    labels, preds, confs = [], [], []
    for x, y in loader:
        probs = torch.softmax(model(x), dim=1)
        c, p = probs.max(dim=1)
        labels.append(y)
        preds.append(p)
        confs.append(c)
    return torch.cat(labels).numpy(), torch.cat(preds).numpy(), torch.cat(confs).numpy()


@torch.no_grad()
def measure_latency(model, loader, n=200, warmup=20):
    """Time ONE picture at a time (like a real device would), in milliseconds."""
    images = torch.cat([x for x, _ in loader])[: n + warmup]
    for i in range(warmup):
        model(images[i:i + 1])
    times = []
    for i in range(warmup, warmup + n):
        t0 = time.perf_counter()
        model(images[i:i + 1])
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return times.mean(), np.percentile(times, 50), np.percentile(times, 99)


def zipped_size_mb(path):
    """Size after zip compression - shows the benefit of zeros in unstructured pruning."""
    tmp = path + ".tmp.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path)
    size = os.path.getsize(tmp) / (1024 * 1024)
    os.remove(tmp)
    return size


def run_group(group_name, names):
    loader = get_test_loader(batch_size=200)
    pred_columns, rows = {}, []
    labels = None

    for name in names:
        path = model_path(name)
        if not os.path.exists(path):
            print(f"[{name}] model file missing, skipped")
            continue
        print(f"\nTesting {name} ...")
        model = load_model(name)
        labels, preds, confs = all_predictions(model, loader)
        accuracy = 100.0 * (preds == labels).mean()
        mean_ms, p50_ms, p99_ms = measure_latency(model, loader)

        is_int8 = name in ("ptq", "qat")
        if is_int8:
            total = nonzero = macs = ""
            sparsity = ""
        else:
            total, nonzero = count_params(model)
            macs = round(count_macs(model) / 1e6, 2)
            sparsity = round(100 * (1 - nonzero / total), 2)

        rows.append({
            "model": name, "accuracy_percent": round(accuracy, 2),
            "file_size_mb": round(file_size_mb(path), 3), "zipped_size_mb": round(zipped_size_mb(path), 3),
            "total_weights": total, "nonzero_weights": nonzero, "sparsity_percent": sparsity,
            "macs_millions": macs,
            "latency_ms_mean": round(mean_ms, 3), "latency_ms_p50": round(p50_ms, 3),
            "latency_ms_p99": round(p99_ms, 3),
            "throughput_img_per_s": round(1000.0 / mean_ms, 1),
        })
        pred_columns[name] = (preds, confs)
        print(f"  accuracy {accuracy:.2f}%   latency {mean_ms:.2f} ms/picture")

    if not rows:
        print(f"No models found for {group_name}.")
        return

    # per-picture predictions
    pred_path = os.path.join(RESULTS_DIR, f"{group_name}_predictions.csv")
    with open(pred_path, "w", newline="") as f:
        header = ["image_index", "true_label", "true_name"]
        for n in pred_columns:
            header += [f"{n}_pred", f"{n}_pred_name", f"{n}_confidence", f"{n}_correct"]
        w = csv.writer(f)
        w.writerow(header)
        for i in range(len(labels)):
            row = [i, int(labels[i]), CLASSES[labels[i]]]
            for n, (p, c) in pred_columns.items():
                row += [int(p[i]), CLASSES[p[i]], f"{c[i]:.4f}", int(p[i] == labels[i])]
            w.writerow(row)

    # summary
    bench_path = os.path.join(RESULTS_DIR, f"{group_name}_benchmark.csv")
    with open(bench_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    print(f"\nSaved {pred_path}\nSaved {bench_path}")
    print(f"{'model':<14}{'acc %':>8}{'size MB':>10}{'ms/img':>9}")
    for r in rows:
        print(f"{r['model']:<14}{r['accuracy_percent']:>8}{r['file_size_mb']:>10}{r['latency_ms_mean']:>9}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--group", choices=["q1", "q2", "all"], default="all")
    args = parser.parse_args()

    set_seed(42)
    torch.set_num_threads(1)   # one CPU thread = fair, repeatable speed comparison
    torch.backends.quantized.engine = get_quant_backend()

    if args.group in ("q1", "all"):
        run_group("q1", Q1_MODELS)
    if args.group in ("q2", "all"):
        run_group("q2", Q2_MODELS)


if __name__ == "__main__":
    main()


Writing benchmark.py


In [10]:
%%writefile onnx_inference.py
"""
onnx_inference.py
-----------------
Run every .onnx file in onnx_models/ on the CPU with ONNX Runtime,
measure accuracy + speed, and write results/onnx_benchmark.csv.

Run:  python onnx_inference.py
      python onnx_inference.py --image my_cat.png --onnx onnx_models/resnet18_fp32.onnx
"""
import argparse
import csv
import glob
import os
import time

import numpy as np
import onnxruntime as ort
from PIL import Image

from common import CLASSES, MEAN, STD, ONNX_DIR, RESULTS_DIR, get_test_loader


def make_session(path):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1       # one thread = fair speed comparison
    return ort.InferenceSession(path, opts, providers=["CPUExecutionProvider"])


def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def predict_image(session, image_path):
    img = Image.open(image_path).convert("RGB").resize((32, 32))
    x = (np.asarray(img, dtype=np.float32) / 255.0 - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    x = x.transpose(2, 0, 1)[None].astype(np.float32)
    probs = softmax(session.run(None, {"image": x})[0])[0]
    return CLASSES[int(probs.argmax())], float(probs.max())


def benchmark_all():
    files = sorted(glob.glob(os.path.join(ONNX_DIR, "*.onnx")))
    if not files:
        print("No .onnx files found. Run export_onnx.py first.")
        return
    loader = get_test_loader(batch_size=200)
    batches = [(x.numpy(), y.numpy()) for x, y in loader]
    rows = []
    for path in files:
        name = os.path.basename(path)
        print(f"\nTesting {name} ...")
        try:
            sess = make_session(path)
            correct = total = 0
            for x, y in batches:
                out = sess.run(None, {"image": x})[0]
                correct += int((out.argmax(1) == y).sum())
                total += len(y)
            single = np.concatenate([b[0] for b in batches])[:220]
            for i in range(20):                                   # warm-up
                sess.run(None, {"image": single[i:i + 1]})
            times = []
            for i in range(20, 220):
                t0 = time.perf_counter()
                sess.run(None, {"image": single[i:i + 1]})
                times.append((time.perf_counter() - t0) * 1000)
            times = np.array(times)
            rows.append({"onnx_file": name, "accuracy_percent": round(100.0 * correct / total, 2),
                         "file_size_mb": round(os.path.getsize(path) / (1024 * 1024), 3),
                         "latency_ms_mean": round(times.mean(), 3),
                         "latency_ms_p50": round(float(np.percentile(times, 50)), 3),
                         "latency_ms_p99": round(float(np.percentile(times, 99)), 3)})
            print(f"  accuracy {rows[-1]['accuracy_percent']}%   latency {rows[-1]['latency_ms_mean']} ms/picture")
        except Exception as err:
            print(f"  FAILED: {type(err).__name__}: {str(err)[:300]}")

    if rows:
        out_path = os.path.join(RESULTS_DIR, "onnx_benchmark.csv")
        with open(out_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        print(f"\nSaved {out_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", help="classify one picture instead of benchmarking")
    parser.add_argument("--onnx", default=os.path.join(ONNX_DIR, "resnet18_fp32.onnx"))
    args = parser.parse_args()
    if args.image:
        label, conf = predict_image(make_session(args.onnx), args.image)
        print(f"Predicted: {label} ({conf * 100:.1f}%)")
    else:
        benchmark_all()


if __name__ == "__main__":
    main()


Writing onnx_inference.py


In [11]:
%env EDGE_WIDTH=64
%env EDGE_TRAIN_IMAGES=50000

env: EDGE_WIDTH=64
env: EDGE_TRAIN_IMAGES=50000


In [12]:
!python q1_train_baseline.py --epochs 10

Training on: cuda
100%|█████████████████████████████████████████| 170M/170M [04:46<00:00, 596kB/s]
Epoch 01/10  loss 1.552  train acc 42.56%  test acc 48.53%  (58s)
Epoch 02/10  loss 1.062  train acc 62.61%  test acc 63.08%  (56s)
Epoch 03/10  loss 0.775  train acc 73.12%  test acc 71.36%  (57s)
Epoch 04/10  loss 0.626  train acc 78.42%  test acc 77.80%  (57s)
Epoch 05/10  loss 0.536  train acc 81.68%  test acc 76.75%  (57s)
Epoch 06/10  loss 0.468  train acc 84.03%  test acc 83.03%  (57s)
Epoch 07/10  loss 0.409  train acc 85.92%  test acc 82.57%  (57s)
Epoch 08/10  loss 0.341  train acc 88.23%  test acc 85.06%  (57s)
Epoch 09/10  loss 0.263  train acc 91.17%  test acc 89.38%  (57s)
Epoch 10/10  loss 0.193  train acc 93.42%  test acc 90.47%  (57s)

Saved FP32 model -> /kaggle/working/edge/models/resnet18_fp32.pth
Training log      -> /kaggle/working/edge/results/train_log_fp32.csv


In [13]:
!python q1_ptq.py

FP32 accuracy: 90.47%
Layers fused.
Quantization backend: fbgemm
/kaggle/working/edge/q1_ptq.py:61: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare(model, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range w

In [14]:
!python q1_qat.py --epochs 6

/kaggle/working/edge/q1_qat.py:51: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(model, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__

In [15]:
!python q2_prune_unstructured.py --rounds 4 --amount 0.35 --ft-epochs 2 --final-epochs 3

Original accuracy: 90.47%
Round 1/4: sparsity 0.0%  accuracy right after pruning 90.43%
          accuracy after fine-tuning 90.16%
Round 2/4: sparsity 0.0%  accuracy right after pruning 88.32%
          accuracy after fine-tuning 90.54%
Round 3/4: sparsity 0.0%  accuracy right after pruning 81.11%
          accuracy after fine-tuning 90.98%
Round 4/4: sparsity 0.0%  accuracy right after pruning 74.50%
          accuracy after fine-tuning 90.14%

FINAL: accuracy 91.43%  sparsity 82.1%  (2,002,516 of 11,173,962 weights are non-zero)
Saved -> /kaggle/working/edge/models/resnet18_unstructured_pruned.pth


In [16]:
!python q2_prune_structured.py --ratio 0.5 --ft-epochs 8

Original: 11,173,962 weights, 555.4 M multiply-adds, accuracy 90.47%
Channels kept in the middle of each block: [32, 32, 64, 64, 128, 128, 256, 256]
Pruned:   5,679,306 weights, 281.7 M multiply-adds (49% fewer)
Accuracy right after pruning (before fine-tuning): 25.53%
Fine-tune epoch 1/8  loss 0.329  test acc 83.98%
Fine-tune epoch 2/8  loss 0.295  test acc 86.86%
Fine-tune epoch 3/8  loss 0.261  test acc 88.05%
Fine-tune epoch 4/8  loss 0.225  test acc 88.65%
Fine-tune epoch 5/8  loss 0.187  test acc 89.74%
Fine-tune epoch 6/8  loss 0.156  test acc 90.72%
Fine-tune epoch 7/8  loss 0.119  test acc 91.35%
Fine-tune epoch 8/8  loss 0.101  test acc 91.80%

FINAL accuracy 91.80%   Saved -> /kaggle/working/edge/models/resnet18_structured_pruned.pth


In [17]:
!python export_onnx.py


FP32 baseline -> ONNX
/kaggle/working/edge/export_onnx.py:34: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, out_path, dynamo=False, **kwargs)   # classic exporter
  saved /kaggle/working/edge/onnx_models/resnet18_fp32.onnx

Unstructured-pruned -> ONNX
  saved /kaggle/working/edge/onnx_models/resnet18_unstructured.onnx

Structured-pruned -> ONNX
  saved /kaggle/working/edge/onnx_models/resnet18_structured.onnx

INT8 with ONNX Runtime static quantization
  saved /kaggle/working/edge/onnx_models/resnet18_int8_ort.onnx

QAT (fake-quant) model -> ONNX
/kaggle/working/edge/export_onnx.py:65: DeprecationWarning: torch.ao.quantizat

In [18]:
!python benchmark.py --group all


Testing fp32 ...
  accuracy 90.47%   latency 18.26 ms/picture

Testing ptq ...
  accuracy 90.49%   latency 8.21 ms/picture

Testing qat ...
  accuracy 91.05%   latency 8.29 ms/picture

Saved /kaggle/working/edge/results/q1_predictions.csv
Saved /kaggle/working/edge/results/q1_benchmark.csv
model            acc %   size MB   ms/img
fp32             90.47    42.701   18.262
ptq              90.49    10.895    8.205
qat              91.05    10.895    8.289

Testing fp32 ...
  accuracy 90.47%   latency 18.39 ms/picture

Testing unstructured ...
  accuracy 91.43%   latency 19.29 ms/picture

Testing structured ...
  accuracy 91.80%   latency 11.43 ms/picture

Saved /kaggle/working/edge/results/q2_predictions.csv
Saved /kaggle/working/edge/results/q2_benchmark.csv
model            acc %   size MB   ms/img
fp32             90.47    42.701   18.391
unstructured     91.43    42.703   19.289
structured        91.8    21.735   11.433


In [19]:
!python onnx_inference.py


Testing resnet18_fp32.onnx ...
  accuracy 90.47%   latency 11.133 ms/picture

Testing resnet18_int8_ort.onnx ...
  accuracy 89.25%   latency 6.493 ms/picture

Testing resnet18_structured.onnx ...
  accuracy 91.8%   latency 5.615 ms/picture

Testing resnet18_unstructured.onnx ...
  accuracy 91.43%   latency 11.213 ms/picture

Saved /kaggle/working/edge/results/onnx_benchmark.csv


In [20]:
!zip -r outputs.zip models results onnx_models

  adding: models/ (stored 0%)
  adding: models/resnet18_qat_int8.pt (deflated 11%)
  adding: models/resnet18_structured_pruned.pth (deflated 7%)
  adding: models/resnet18_ptq_int8.pt (deflated 11%)
  adding: models/resnet18_qat_int8_state_dict.pth (deflated 11%)
  adding: models/resnet18_unstructured_pruned.pth (deflated 73%)
  adding: models/resnet18_ptq_int8_state_dict.pth (deflated 11%)
  adding: models/resnet18_fp32.pth (deflated 7%)
  adding: models/resnet18_qat_fakequant_state_dict.pth (deflated 8%)
  adding: results/ (stored 0%)
  adding: results/q1_predictions.csv (deflated 83%)
  adding: results/train_log_fp32.csv (deflated 40%)
  adding: results/q1_benchmark.csv (deflated 39%)
  adding: results/q2_predictions.csv (deflated 83%)
  adding: results/q2_benchmark.csv (deflated 42%)
  adding: results/onnx_benchmark.csv (deflated 43%)
  adding: onnx_models/ (stored 0%)
  adding: onnx_models/resnet18_fp32.onnx (deflated 7%)
  adding: onnx_models/resnet18_structured.onnx (deflated 7%)

In [21]:
from IPython.display import FileLink
FileLink('/kaggle/working/edge/outputs.zip')

/kaggle/working/edge/outputs.zip

In [22]:
import os, shutil, glob
from IPython.display import FileLink

SRC = "/kaggle/working/edge/onnx_models"      # where export_onnx.py saved them
DST = "/kaggle/working/onnx_models_saved"     # new folder to keep them in

# 1) Show what exists
files = sorted(glob.glob(os.path.join(SRC, "*.onnx")))
print("Found", len(files), "ONNX files:")
for f in files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.2f} MB)")

# 2) Copy them into a new folder
os.makedirs(DST, exist_ok=True)
for f in files:
    shutil.copy(f, DST)

# 3) Zip the folder so you can download it in one click
shutil.make_archive("/kaggle/working/onnx_models", "zip", DST)
print("\nSaved in:", DST)
print("Zip:", "/kaggle/working/onnx_models.zip")

FileLink("/kaggle/working/onnx_models.zip")

Found 4 ONNX files:
  resnet18_fp32.onnx  (44.69 MB)
  resnet18_int8_ort.onnx  (11.29 MB)
  resnet18_structured.onnx  (22.71 MB)
  resnet18_unstructured.onnx  (44.69 MB)

Saved in: /kaggle/working/onnx_models_saved
Zip: /kaggle/working/onnx_models.zip


/kaggle/working/onnx_models.zip